# 🚀 Step 4: Model Training

This notebook trains the LightFaceNet model on the LFW dataset using triplet loss.

## Training Configuration
- **Model**: LightFaceNet (MobileNetV3-Small backbone)
- **Loss**: Triplet Loss (margin=0.2)
- **Optimizer**: Adam (lr=1e-4)
- **Epochs**: 30
- **Batch Size**: 32

In [ ]:
import os
import sys
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Add project root
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
DATA_DIR = PROJECT_ROOT / "data" / "lfw" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4.1 Define Model and Loss

In [ ]:
class LightFaceNet(nn.Module):
    """Lightweight Face Embedding Network."""
    
    def __init__(self, embedding_dim=128, backbone="mobilenet_v3_small", pretrained=True, dropout=0.2):
        super().__init__()
        
        self.embedding_dim = embedding_dim
        
        if backbone == "mobilenet_v3_small":
            base = models.mobilenet_v3_small(weights='IMAGENET1K_V1' if pretrained else None)
            feature_dim = 576
        elif backbone == "mobilenet_v3_large":
            base = models.mobilenet_v3_large(weights='IMAGENET1K_V1' if pretrained else None)
            feature_dim = 960
        else:
            raise ValueError(f"Unknown backbone: {backbone}")
        
        self.features = base.features
        self.avgpool = base.avgpool
        
        self.embedding = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, embedding_dim)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.embedding(x)
        x = F.normalize(x, p=2, dim=1)
        return x


class TripletLoss(nn.Module):
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin
        
    def forward(self, anchor, positive, negative):
        pos_dist = (anchor - positive).pow(2).sum(dim=1)
        neg_dist = (anchor - negative).pow(2).sum(dim=1)
        loss = F.relu(pos_dist - neg_dist + self.margin)
        return loss.mean()


print("✅ Model and loss defined")

## 4.2 Create Dataset

In [ ]:
class TripletFaceDataset(Dataset):
    """Dataset that generates triplets for training."""
    
    def __init__(self, images, labels, triplets_per_image=2):
        self.images = images
        self.labels = labels
        self.triplets_per_image = triplets_per_image
        
        # Build label index
        self.label_to_indices = {}
        for idx, label in enumerate(labels):
            if label not in self.label_to_indices:
                self.label_to_indices[label] = []
            self.label_to_indices[label].append(idx)
        
        # Valid labels (at least 2 images)
        self.valid_labels = [l for l, indices in self.label_to_indices.items() if len(indices) >= 2]
        self.all_labels = list(self.label_to_indices.keys())
        
    def __len__(self):
        return len(self.images) * self.triplets_per_image
    
    def __getitem__(self, idx):
        import random
        
        anchor_idx = idx % len(self.images)
        anchor_label = self.labels[anchor_idx]
        
        # Handle labels with only 1 image
        if anchor_label not in self.valid_labels:
            anchor_label = random.choice(self.valid_labels)
            anchor_idx = random.choice(self.label_to_indices[anchor_label])
        
        # Get positive
        positive_indices = [i for i in self.label_to_indices[anchor_label] if i != anchor_idx]
        positive_idx = random.choice(positive_indices)
        
        # Get negative
        negative_label = random.choice([l for l in self.all_labels if l != anchor_label])
        negative_idx = random.choice(self.label_to_indices[negative_label])
        
        # Load and convert images
        anchor = torch.from_numpy(self.images[anchor_idx].transpose(2, 0, 1)).float()
        positive = torch.from_numpy(self.images[positive_idx].transpose(2, 0, 1)).float()
        negative = torch.from_numpy(self.images[negative_idx].transpose(2, 0, 1)).float()
        
        return anchor, positive, negative


print("✅ Dataset class defined")

## 4.3 Load Data

In [ ]:
# Load processed data
print("Loading data...")

train_images = np.load(DATA_DIR / "train_images.npy")
train_labels = np.load(DATA_DIR / "train_labels.npy")
val_images = np.load(DATA_DIR / "val_images.npy")
val_labels = np.load(DATA_DIR / "val_labels.npy")

print(f"✅ Loaded data:")
print(f"   Train: {len(train_images)} images")
print(f"   Val: {len(val_images)} images")

# Create datasets and loaders
BATCH_SIZE = 32

train_dataset = TripletFaceDataset(train_images, train_labels, triplets_per_image=2)
val_dataset = TripletFaceDataset(val_images, val_labels, triplets_per_image=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n✅ Created loaders:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

## 4.4 Training Configuration

In [ ]:
# Hyperparameters
CONFIG = {
    "epochs": 30,
    "batch_size": BATCH_SIZE,
    "learning_rate": 1e-4,
    "margin": 0.2,
    "embedding_dim": 128,
    "backbone": "mobilenet_v3_small",
}

# Create model
model = LightFaceNet(
    embedding_dim=CONFIG["embedding_dim"],
    backbone=CONFIG["backbone"],
    pretrained=True
).to(device)

# Loss, optimizer, scheduler
loss_fn = TripletLoss(margin=CONFIG["margin"])
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)

print("\n📊 Training Configuration")
print("=" * 40)
for k, v in CONFIG.items():
    print(f"   {k}: {v}")
print(f"\n   Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4.5 Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    
    for anchor, positive, negative in tqdm(loader, desc="Training", leave=False):
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)
        
        optimizer.zero_grad()
        
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)
        
        loss = loss_fn(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def validate_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for anchor, positive, negative in tqdm(loader, desc="Validating", leave=False):
            anchor = anchor.to(device)
            positive = positive.to(device)
            negative = negative.to(device)
            
            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)
            
            loss = loss_fn(emb_a, emb_p, emb_n)
            total_loss += loss.item()
    
    return total_loss / len(loader)

In [ ]:
# Training
print("\n" + "=" * 50)
print("🚀 Starting Training")
print("=" * 50)

history = {"train_loss": [], "val_loss": [], "lr": []}
best_val_loss = float("inf")

for epoch in range(1, CONFIG["epochs"] + 1):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    
    # Validate
    val_loss = validate_epoch(model, val_loader, loss_fn, device)
    
    # Update scheduler
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()
    
    # Record history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)
    
    # Print progress
    improved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save best model
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "config": CONFIG
        }, MODELS_DIR / "best_model.pth")
        improved = " ✓ (saved)"
    
    print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.2e}{improved}")

print("\n✅ Training complete!")
print(f"   Best validation loss: {best_val_loss:.4f}")

## 4.6 Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history["train_loss"], label="Train Loss", marker='o', markersize=3)
axes[0].plot(history["val_loss"], label="Val Loss", marker='s', markersize=3)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Triplet Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LR plot
axes[1].plot(history["lr"], color='green', marker='o', markersize=3)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule")
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / "training_history.png", dpi=150)
plt.show()

print(f"📊 Training plot saved to {MODELS_DIR / 'training_history.png'}")

## 4.7 Save Final Model and Config

In [ ]:
# Save final model
torch.save({
    "epoch": CONFIG["epochs"],
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "history": history
}, MODELS_DIR / "final_model.pth")

# Save config
training_info = {
    **CONFIG,
    "best_val_loss": best_val_loss,
    "final_train_loss": history["train_loss"][-1],
    "final_val_loss": history["val_loss"][-1],
    "device": device,
    "timestamp": datetime.now().isoformat()
}

with open(MODELS_DIR / "training_config.json", "w") as f:
    json.dump(training_info, f, indent=2)

print("\n📁 Saved files:")
for f in MODELS_DIR.glob("*"):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name}: {size_mb:.1f} MB")

## ✅ Step 4 Complete!

**Summary:**
- Trained LightFaceNet for {CONFIG['epochs']} epochs
- Best validation loss: {best_val_loss:.4f}
- Model saved to `models/best_model.pth`

**👉 Next Step:** Run `05_model_evaluation.ipynb` to evaluate on LFW benchmark